<a href="https://colab.research.google.com/github/gustkt/GE338-Lab-2/blob/main/lab2_spatial_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Lab 2: Spatial Analysis กับ Google Earth Engine**

**Situation**

หน่วยงานจัดการทรัพยากรธรรมชาติของจังหวัดแห่งหนึ่งในภาคเหนือของไทยต้องการข้อมูลพื้นฐานเพื่อวางแผนการจัดการที่ดินในฤดูแล้ง พวกเขาต้องการทราบว่า

• พื้นที่ไหนมีพืชพรรณหนาแน่น และมีแนวโน้มเปลี่ยนแปลงหรือไม่

• บริเวณใดมีความชื้นสะสม หรือมีแหล่งน้ำผิวดิน

• ข้อมูลเชิงพื้นที่เหล่านี้ควรจัดเก็บและส่งต่ออย่างไรเพื่อให้ใช้งานได้จริง

* **ภารกิจที่ 1: เลือกพื้นที่ศึกษาและเหตุผล**

**พื้นที่ศึกษา : จ.ลำพูน**

เนื่องจากจ.ลำพูนเป็นพื้นที่เกษตรกรรมสำคัญในภาคเหนือโดยเฉพาะการปลูกลำไยซึ่งมีความอ่อนไหวต่อความแห้งแล้ง นอกจากนี้ยังมีพื้นที่ที่ตั้งอยู่ในบริเวณลุ่มน้ำปิงทำให้มีความหลากหลายของสภาพความชื้นทั้งพื้นที่ชุ่มน้ำและพื้นที่แห้งแล้งในช่วงเวลาเดียวกัน ด้วยเหตุผลที่กล่าวไปข้างต้นจ.ลำพูนจึงเหมาะสมในการวิเคราะห์ความหนาแน่นของพืชพรรณ และสภาพความชื้นเพื่อสนับสนุนวางแผนการจัดการที่ดินในฤดูแล้งต่อไป

* **ภารกิจที่ 2: เลือกดาวเทียมและช่วงเวลา**

* ดาวเทียมที่เลือกใช้ : Harmonized Sentinel-2 MSI: MultiSpectral Instrument, Level-2A (SR) เนื่องจาก Band Red, Green, Blue, NIR ที่สำคัญกับการแยกพื้นที่เกษตรและน้ำมี spatial resolution สูงกว่า Landsat (10 m) รวมไปถึง revisit time ที่ถี่กว่า (5 วัน) ทำให้สามารถเลือกภาพได้หลากหลายวันมากกว่า

* ช่วงเวลา : เดือนธันวาคม 2025 เพราะเป็นช่วงฤดูแล้งตรงกับที่ต้องการข้อมูลพื้นฐานเพื่อวางแผนการจัดการที่ดินต่อไป

* การตั้งเกณฑ์กรองเมฆ : ตั้งไว้ที่ < 20% เพราะถ้าหากตั้งเกณฑ์ต่ำไป เช่น < 10% คุณภาพของภาพจะดีมากเนื่องจากมีเมฆน้อย แต่จำนวนภาพก็น้อยมากเช่นกันทำให้เสื่องไม่มีภาพพอในบางพื้นที่ หรือบางช่วงเวลา ในขณะที่หากตั้งเกณฑ์สูงไป เช่น < 50% จะได้ภาพจำนวนมากก็จริงแต่มีเมฆปนเยอะ ทำให้เมื่อนำไปคำนวณดัชนีต่าง ๆ อาจทำให้ค่าที่ได้ผิดเพี้ยน ดังนั้นการเกณฑ์ตั้งไว้ที่ < 20% เป็นการ balance ระหว่างคุณภาพและการมีข้อมูลของภาพถ่ายดาวเทียม

In [ ]:
# ติดตั้ง + import
!pip install -q geemap

In [ ]:
import ee
import geemap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
# Authenticate + Initialize
ee.Authenticate()
ee.Initialize(project='ee-gustkt45513')

In [ ]:
# ดึงขอบเขตจังหวัดลำพูน จาก GEE

# ใช้ FAO GAUL dataset
roiPT = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM1_NAME', 'Lamphun'))

# ฟังก์ชัน mask cloud (SCL)
def mask_s2_clouds_scl(image):
    scl = image.select('SCL')

# คลาสที่เราต้องการ "กรองออก" (Mask out)
  # 3: Cloud Shadow
  # 8: Cloud Medium Probability
  # 9: Cloud High Probability
  # 10: Thin Cirrus
  # 11: Snow
    bad_classes = [3, 8, 9, 10]

    mask = scl.remap(
        bad_classes,
        ee.List.repeat(0, len(bad_classes)),
        1
    )

    return image.updateMask(mask).divide(10000)

# โหลด Sentinel-2
dataset = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2025-12-01', '2025-12-31')  # ฤดูแล้ง
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .filterBounds(roiPT)
    .map(mask_s2_clouds_scl)
    .select(['B2','B3','B4','B8','B11'])
)

image = dataset.median().clip(roiPT)

# แสดง Map
Map = geemap.Map()

# แสดงภาพ RGB
Map.addLayer(image, {
    'min': 0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2']
}, 'Sentinel-2 RGB')

# แสดงขอบเขต ROI
Map.addLayer(roiPT, {}, 'ROI')

# zoom ไปที่พื้นที่
Map.centerObject(roiPT, 8)

Map

* **ภารกิจที่ 3: วิเคราะห์และตีความ**

* คำนวณและแสดงผล Spectral Index ที่เหมาะสมกับพื้นที่ศึกษา

In [ ]:
# เลือกคำนวณ NDVI, NDWI และ NDBI
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')

# รวมเป็น stack
stack = image.addBands([ndvi, ndwi, ndbi])

# แสดงแผนที่
Map.addLayer(ndvi, {'min': -1, 'max': 1, 'palette': ['blue','white','green']}, 'NDVI')
Map.addLayer(ndwi, {'min': -1, 'max': 1, 'palette': ['brown','white','blue']}, 'NDWI')
Map.addLayer(ndbi, {'min': -1, 'max': 1, 'palette': ['green','white','red']}, 'NDBI')
Map

จากการคำนวณ NDVI, NDWI และ NDBI พบว่าใน layer NDVI แสดงเป็นสีเขียวกระจายทั่วพื้นที่จังหวัดสอดคล้องกับลักษณะภูมิประเทศที่เต็มไปด้วยป่าไม้ทำให้ layer NDWI แสดงผลในลักษณะเดียวกัน เนื่องจากป่าไม้มักมีความชุ่มชื้นสูงกว่าพื้นที่อื่น และ layer NDBI ที่เป็น index แสดงพื้นที่เมืองทำให้เห็นการตั้งถิ่นฐาน และเส้นทางคมนาคมในจังหวัดลำพูนอย่างชัดเจน

* สร้าง Zonal Statistics สำหรับหน่วยบริหาร (อำเภอ/ตำบล) ของพื้นที่

In [ ]:
# Zonal Statistics (อำเภอ)

zonal_stats = stack.reduceRegions(
    collection=districts,
    reducer=ee.Reducer.mean(),
    scale=10
)

# Clean table (สำหรับ CSV)

def clean_feature(f):
    return ee.Feature(None, {
        'Province': f.get('ADM1_NAME'),
        'District': f.get('ADM2_NAME'),
        'NDVI': f.get('NDVI'),
        'NDWI': f.get('NDWI'),
        'NDBI': f.get('NDBI')
    })

zonal_clean = zonal_stats.map(clean_feature)

# ดึงข้อมูล
features = zonal_clean.toList(zonal_clean.size())
data = features.getInfo()

# แปลงเป็น DataFrame
df = pd.DataFrame([f['properties'] for f in data])

pd.set_option('display.max_rows', None)

# แสดงผล
df

In [ ]:
df.nlargest(3, 'NDVI')

In [ ]:
df.nlargest(3, 'NDWI')

In [ ]:
df.nlargest(3, 'NDBI')

จาก Zonal Statistic รายอำเภอ พบว่าอำเภอที่มี NDBI สูงสุด 3 แรก ได้แก่ อำเภอแม่ทา (Mae Tha), อำเภอบ้านโฮ่ง (Ban Hong) และอำเภอทุ่งหัวช้าง (Thung Hua Chang)	 ส่วน 3 อันดับแรกที่มี NDVI และ NDWI มากสุด ได้แก่ อำเภอบ้านธิ (Ban Thi), อำเภอเมืองลำพูน (Muang Lamphun)	และอำเภอป่าซาง (Pa Sang)

* วิเคราะห์ความสัมพันธ์ระหว่าง Index 2 ตัว (NDVI vs NDWI) และ Interpret ผล

In [ ]:
# สุ่ม pixel จาก stack (NDVI, NDWI)
sample = stack.select(['NDVI','NDWI']).sample(
    region=roiPT,
    scale=10,
    numPixels=5000,
    geometries=False
)

In [ ]:
# แปลงเป็น pandas DataFrame
data = sample.getInfo()
df = pd.DataFrame([f['properties'] for f in data['features']])

# ลบค่า missing กัน error
df = df.dropna()

df.head()

In [ ]:
# สร้าง Scatter plot + Regression + Correlation
x = df['NDVI']
y = df['NDWI']

# correlation
corr = x.corr(y)

# regression line
m, b = np.polyfit(x, y, 1)

plt.figure()
plt.scatter(x, y)

# เส้น regression
plt.plot(x, m*x + b)

plt.xlabel('NDVI')
plt.ylabel('NDWI')
plt.title(f'NDVI vs NDWI (r = {corr:.2f})')

plt.show()

In [ ]:
# แสดงค่าสถิติ
print("=== Descriptive Statistics ===")
print(df.describe())

print("\nCorrelation (NDVI vs NDWI):", corr)

In [ ]:
# แสดง Distribution
plt.figure()
plt.hist(df['NDVI'], bins=30)
plt.title('NDVI Distribution')
plt.xlabel('NDVI')
plt.ylabel('Frequency')
plt.show()

plt.figure()
plt.hist(df['NDWI'], bins=30)
plt.title('NDWI Distribution')
plt.xlabel('NDWI')
plt.ylabel('Frequency')
plt.show()

จากการวิเคราะห์ความสัมพันธ์ระหว่าง NDVI และ NDWI พบว่ามีความสัมพันธ์แบบแปรผันกัน โดยมีค่า correlation r = -0.96 ซึ่งแสดงให้เห็นว่าพื้นที่ที่มีความเขียวของพืชพรรณสูง (์NDVI สูง) จะมีค่าความชื้นพื้นผิวต่ำ (NDWI ต่ำ) ในช่วงเวลาที่ศึกษา ผลลัพธ์ที่ได้สะท้อนลักษณะพื้นที่ในช่วงฤดูแล้งซึ่งพืชพรรณยังคงมีความเขียว แต่ปริมาณความชื้นลดลงอย่างชัดเจน

* **ภารกิจที่ 4: Export และจัดการข้อมูล**

In [ ]:
# Export FeatureCollection ไปยัง Google Drive (GeoTIFF format)

task = ee.batch.Export.image.toDrive(
    image=image,
    description='Lamphun_S2_Median',
    folder='Data_for_data_science',
    fileNamePrefix='lamphun_s2_median',
    region=roiPT.geometry(),
    scale=10,
    crs='EPSG:32647',  # UTM Zone 47N
    maxPixels=1e13
)

task.start()

CRS ที่เลือก : UTM Zone 47N (EPSG:32647) เนื่องจากมีหน่วยเป็นเมตร สามารถนำไปใช้วัดระยะ หรือพื้นที่ได้แม่นยำ ซึ่งเหมาะกับงานวิเคราะห์เชิงพื้นที่

Scale ที่เลือก : เลือกใช้ scale = 10 เมตร ซึ่งเป็นความละเอียดสูงสุดสำหรับ Spectral ที่ใช้ในการวิเคราะห์ (B3, B4, B8) เพื่อให้แสดง details ของพื้นที่พืชพรรณ และพื้นที่เมืองได้อย่างชัดเจน

Trade-off : เนื่องจากภาพมีความละเอียดสูง (10 เมตร) ทำให้ไฟล์มีขนาดใหญ่ และ ใช้เวลาในการประมวลผลมากกว่า product ที่มีความละเอียด 20 หรือ 30 เมตรซึ่งขนาดไฟล์เล็กกว่า ใช้เวลาในการประมวลผลน้อยกว่าแต่ก็จะสูญเสียรายละเอียดเชิงพื้นที่บางส่วน

Format ของไฟล์ : เลือก export เป็น GeoTIFF (default ของ GEE) เพราะเป็น standard format สำหรับ Raster ที่สามาถเก็บค่าพิกัดภูมิศาสตร์ (georeferencing) ได้ครบถ้วน และรองรับการใช้งาน/วิเคราะห์ผ่านโปรแกรมทาง GIS

